[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NguyenVu04/band-tilt/blob/main/notebooks/03b_morbo.ipynb)

# 03b — Multi-Objective Bayesian Optimization with MORBO

## 1. Overview

This notebook runs **MORBO**, the proposed method for multi-band antenna tilt coordination, and reads it against the
baseline of notebook 03a.

**Question.** Does a model-guided multi-objective search find better trade-offs between coverage, separation and
throughput than random search, for the same number of ray-traced evaluations?

**Goals.**

- Run MORBO on the same simulator, scenario, objectives and evaluation budget as random search.
- Show how its trust region spends that budget and what front it finds.
- Measure what the recommended configuration changes on every KPI, and where on the map, beside random search.

The problem formulation is described in notebook 01, the setup tables and the objectives in notebook 03a
(sections 3 and 7); they are not repeated here.

**Outputs.** One run directory per seed under `outputs/optim/morbo/`, tables under `reports/tables/03b_morbo/`,
figures under `reports/figures/03b_morbo/`. **Requirements.** A CUDA GPU, and notebook 03a run first. The shell
equivalent of the run is `task bo`.


In [ ]:
# Environment: locally, move to the project root; on Colab, clone the repository
# and install what Colab lacks. Extra Hydra overrides come from BAND_TILT_OVERRIDES.
REPO_URL = "https://github.com/NguyenVu04/band-tilt.git"
COLAB_PACKAGES = [("hydra", "hydra-core"), ("sionna.rt", "sionna-rt"), ("botorch", "botorch")]

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    root = Path("/content/band-tilt")
    if not root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)
    missing = [pip for module, pip in COLAB_PACKAGES if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    root = Path.cwd()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent

os.chdir(root)
sys.path.insert(0, str(root))
CONFIG_OVERRIDES = os.environ.get("BAND_TILT_OVERRIDES", "").split()

In [ ]:
%load_ext autoreload
%autoreload 2

from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from src.config import load_config
from src.core.sector import read_sectors, site_frame
from src.evaluation import compare, plots
from src.evaluation import runs as run_store
from src.evaluation.export import readable, save_table
from src.kpi.capacity import max_rsrp
from src.optim.methods.morbo.search import MORBO, RESTART, failure_tolerance, local_size
from src.optim.objective import OBJECTIVE_NAMES, hypervolume_contributions, pareto_mask
from src.optim.run import run
from src.optim.space import TiltSpace
from src.utils.plotting import label, save_fig, setup_plotting

cfg = load_config(overrides=CONFIG_OVERRIDES)
setup_plotting()
save_fig = partial(save_fig, in_colab=IN_COLAB, directory=Path(cfg.reports.figures_dir) / "03b_morbo")
save_table = partial(save_table, in_colab=IN_COLAB, directory=Path(cfg.reports.tables_dir) / "03b_morbo")
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 40)

# Search seeds, matched to 03a: optim.seed from the config; BAND_TILT_SEEDS (space-separated) overrides it.
SEEDS = [int(seed) for seed in os.environ.get("BAND_TILT_SEEDS", str(cfg.optim.seed)).split()]
METHODS = ("morbo", "random")


## 2. Method

MORBO (Daulton et al., UAI 2022, `optim/method=morbo`) keeps one or more trust regions, boxes in the
unit cube the tilt box is rescaled to, and searches inside them for points that add hypervolume:

1. Evaluate the current configuration, then an initial Sobol design of `n_init` points, the same design random search
   starts with.
2. Centre each region on the Pareto point with the largest hypervolume contribution it may take: inside the region
   if one is free, else anywhere. Another region's centre and a collapsed region's old centre are taken.
3. Fit one GP per objective (constant mean, Matérn-5/2 kernel with ARD under BoTorch's dimension-scaled lengthscale
   prior, standardized outcome) to every evaluation
   within a cube of side $2L$ around the centre, topped up with the nearest points to at least $\min(250, 2d)$, the
   inputs rescaled from that cube. Evaluations are shared: a region's model sees what any region found.
4. For each point of the batch, draw `n_candidates` fresh candidates per region by perturbing random Pareto points
   inside it, each dimension with probability $p_n = p_0\bigl(1 - 0.5\log(n - n_0 + 1) / \log(N - n_0 + 1)\bigr)$,
   $p_0 = \min(k/d, 1)$, $k$ = `perturbed_dimensions`.
5. Pick a batch of `batch_size` one point at a time: draw a joint posterior sample over each region's candidates and
   the points already picked inside it, and take the candidate whose sample adds the most hypervolume to the observed
   front and those picked points (BoTorch's `FastNondominatedPartitioning` turns that into one vectorized computation
   per draw). When no candidate adds any, a random scalarisation of the objectives breaks the tie.
6. Ray-trace the batch. A region's points succeed when they raise the hypervolume by the share `improvement`;
   after $\max(10, \lceil d/3 \rceil)$ failed evaluations in a row its side halves. Regions never grow. Below
   `length_min` a region restarts, at a side that decays with the budget used, around the point a random hypervolume
   scalarisation ranks best in one draw of a global GP over the initial design and earlier restart points.

Every proposal is snapped to the tilt resolution before it is traced. Every candidate is ray-traced and scored exactly
as in 03a; the GPs only choose where to look, and no reported number comes from them.

## 3. Configuration

In [ ]:
space = TiltSpace.from_config(cfg)
baseline = run_store.baseline_map(cfg)
sectors = site_frame(read_sectors(cfg.simulation.input.sectors_file))
ue = pd.read_parquet(cfg.data.output.ue_file)

morbo_cfg = load_config(overrides=["optim/method=morbo", *CONFIG_OVERRIDES])
method = morbo_cfg.optim.method
region = method.trust_region
settings = pd.DataFrame(
    {
        "Parameter": [
            "Optimized variables",
            "Tilt resolution [°]",
            "Evaluations (excluding the incumbent)",
            "Budget split",
            "Batch size",
            "Trust regions",
            "Trust region length: initial, minimum",
            "Failure tolerance (evaluations)",
            "Improvement for a success (share of the hypervolume)",
            "Perturbed dimensions per candidate at the start (expected)",
            "Candidates per region and round",
            "Fewest points per local model",
            "Kernel",
            "Hypervolume reference point",
            "Search seeds",
        ],
        "Value": [
            space.n_dim,
            f"{space.resolution_deg:g}",
            int(method.budget.n_init) + int(method.budget.n_iter),
            f"n_init={method.budget.n_init}, n_iter={method.budget.n_iter}",
            int(method.budget.batch_size),
            int(region.count),
            f"{float(region.length_init):g}, {float(region.length_min):g}",
            failure_tolerance(space.n_dim),
            f"{float(region.improvement):g}",
            f"{min(float(region.perturbed_dimensions), space.n_dim):g} of {space.n_dim}",
            int(method.n_candidates),
            local_size(space.n_dim),
            "Matérn-5/2 with ARD, one GP per objective",
            "origin (0, 0)",
            ", ".join(map(str, SEEDS)),
        ],
    }
)
save_table(settings, "morbo_configuration")
settings


## 4. Runs

A MORBO run already on disk for the same seed and the current scenario is reused; delete its directory under
`outputs/optim/morbo/` to repeat it. Runs from an earlier scenario are ignored. The baseline runs are read from disk,
not repeated, and every run is checked comparable with the current radio map.


In [ ]:
scenario = str(baseline["scenario_id"])


def on_disk():
    # The newest finished run of each method and seed on the current scenario.
    current = [r for r in run_store.discover(cfg.optim.output.dir) if r.scenario_id == scenario]
    return run_store.latest_per_method_and_seed(current)


done = {(r.method, r.seed) for r in on_disk()}
for seed in SEEDS:
    if ("morbo", seed) not in done:
        run(load_config(overrides=["optim/method=morbo", f"optim.seed={seed}", *CONFIG_OVERRIDES]))

runs = [r for r in on_disk() if r.method in METHODS and r.seed in SEEDS]
missing = [m for m in METHODS if not any(r.method == m for r in runs)]
if missing:
    raise run_store.RunError(
        f"No {', '.join(missing)} run on scenario {scenario}; run notebook 03a first."
    )
checks = run_store.verify(runs, baseline)
run_store.require(checks)
print(f"{len(runs)} runs; {int(checks['holds'].sum())} of {len(checks)} comparability checks hold")
best = compare.best_run_per_method(runs)
morbo = best["morbo"]


## 5. Search Progress

Hypervolume of every evaluation so far, for MORBO and random search, then how much each search added beyond its
initial design. With several seeds the line is the mean and the band spans the best and worst seed.


In [ ]:
figure = plots.convergence_plot(compare.convergence(runs))
save_fig(figure, "search_progress")
plt.show()

volumes = readable(compare.hypervolume_table(runs))
save_table(volumes, "hypervolume")
volumes


### Where MORBO spent its budget

Every evaluation of MORBO's best run on each objective, marked by what proposed it: the Sobol design, a trust-region
proposal, or a region's restart point.


In [ ]:
history = morbo.history
proposer = history["generation_node"].replace({"attached": "Current configuration"})
colours = {
    "Current configuration": plots.COLOURS["incumbent"],
    "Sobol": plots.COLOURS["random"],
    MORBO: plots.COLOURS["morbo"],
    RESTART: "tab:purple",
}
values = compare.objective_values(history)
on_front = pareto_mask(values)
restarts = int(((history["phase"] == "init") & (history["iteration"] > int(method.budget.n_init))).sum())

figure, axes = plt.subplots(
    1, len(OBJECTIVE_NAMES), figsize=(5.4 * len(OBJECTIVE_NAMES), 4.6), constrained_layout=True
)
for axis, name in zip(axes, OBJECTIVE_NAMES, strict=True):
    for key, group in history.groupby(proposer, sort=False):
        axis.scatter(group["iteration"], group[name], s=14, color=colours[key], label=key)
    axis.scatter(
        history.loc[on_front, "iteration"], history.loc[on_front, name], s=60,
        facecolors="none", edgecolors="black", label="Pareto front",
    )
    axis.set(xlabel="Evaluations", ylabel=f"{label(name)} (maximise)")
axes[0].legend(fontsize=8)
figure.suptitle(f"MORBO, seed {morbo.seed}: every evaluation, {restarts} restarts")
save_fig(figure, "morbo_evaluations")
plt.show()

by_proposer = history.assign(pareto=on_front).groupby(proposer, sort=False).agg(
    Evaluations=("iteration", "size"),
    **{f"Mean {label(name).lower()}": (name, "mean") for name in OBJECTIVE_NAMES},
    **{"On the Pareto front": ("pareto", "sum")},
).rename_axis("Proposed by")
save_table(by_proposer.reset_index(), "morbo_evaluations_by_proposer")
by_proposer


**Observations.** Both searches share the 8-point Sobol design, which takes the hypervolume from today's 0.3514 to
0.3735. MORBO passes random search's final hypervolume (0.3754) at evaluation 14 and keeps adding to it until
evaluation 70, ending at 0.4075; random search's last gain is at evaluation 60.

**The model is steering.** MORBO's 64 proposals average coverage 0.6299 and separation 0.6017 (throughput 3.3453),
against 0.5659, 0.5830 and 3.2787 for the Sobol design, and 10 of them beat today's configuration on both searched
objectives, which no random-search candidate does. Every one of the 8 front points is a trust-region proposal. The
trust region never collapsed — the run ended on a failure streak of 10 of the 12 evaluations that halve it — so no
restart was spent.

## 6. Pareto Front

Every configuration both searches evaluated, on the pair of objectives; the stars are each method's recommended
point and the dashed line the two-objective front. Then MORBO's own front, every point it found that no other point
it evaluated beats on both objectives, with its hypervolume contribution.


In [ ]:
searched = compare.candidates(runs)
for x, y in (
    ("coverage_objective", "separation_objective"),
):
    figure = plots.tradeoff_scatter(searched, x, y)
    save_fig(figure, f"tradeoff_{x}_vs_{y}")
    plt.show()

front = history.loc[on_front, ["iteration", "phase", *OBJECTIVE_NAMES]].assign(
    hypervolume_contribution=hypervolume_contributions(values)[on_front],
    recommended=lambda f: f["iteration"] == morbo.best_index,
).sort_values("hypervolume_contribution", ascending=False)
front = readable(front.rename(columns={"hypervolume_contribution": "Hypervolume contribution"}))
save_table(front, "morbo_pareto_front")
front


**Observations.** MORBO's front holds eight points, all trust-region proposals. The recommendation rule, the largest
hypervolume contribution against the origin, picks **evaluation 21**: coverage 0.6150 and separation 0.6224, both
above today's 0.5743 and 0.6118, the best separation on the front. Its contribution, 0.00194, is five times the
next one (evaluation 62, 0.00036), which trades separation (0.6051, below today) for the most coverage (0.6552).

Four front points beat today on both objectives (evaluations 21, 27, 45 and 69); the other four buy coverage with
separation below today's. All eight are published, the recommendation first (`reports/outputs/solutions_morbo.csv`).

## 7. Recommended Tilt Configuration

The deliverable of MORBO's best run: current and proposed tilt per sector-band. Negative change is uptilt.


In [ ]:
tilt = readable(morbo.best_tilt)
save_table(tilt, "best_tilt_morbo")
summary = readable(compare.tilt_movement(morbo))
save_table(summary, "tilt_movement_morbo")
print(f"{label('morbo')}: run {morbo.run_id}, seed {morbo.seed}, recommended evaluation {morbo.best_index}")
display(summary)
display(tilt)
figure = plots.tilt_movement_plot(morbo.best_tilt, "morbo")
save_fig(figure, "tilt_movement_morbo")
plt.show()


**Observations.** The recommendation moves 32 of the 36 sector-bands, 20 up and 12 down, to between 0.1° and 17.6°.
Net it uptilts 700 MHz by 2.7° and leaves 2600 MHz (−0.3°) and 1800 MHz (+0.2°) about where they were. The largest
single change is 9.9° (n1s2, 2600 MHz, to 0.1°).

The downtilts sit on seven sectors and often pair with an uptilt on another band of the same sector: n0s1 pulls
2600 MHz (+7.2°) and 1800 MHz (+7.4°) in while its 700 MHz carrier uptilts 9.1°; n3s0 pulls 1800 MHz in by 2.6°
while 700 MHz uptilts 6.6°; n0s2 pulls all three bands in. With one seed it is not established which of these
per-sector choices matter.

## 8. Final Results

### KPI comparison

Every KPI and objective of the current configuration and of each method's recommended configuration, on every UE,
then what each run found and what it cost.


In [ ]:
incumbent = morbo.incumbent_kpi
comparison = pd.DataFrame(
    {
        "kpi": list(compare.SEARCH_MEASURES),
        "direction": [compare.direction(name) for name in compare.SEARCH_MEASURES],
        "incumbent": [getattr(incumbent, name) for name in compare.SEARCH_MEASURES],
        **{m: [getattr(best[m].best_kpi, name) for name in compare.SEARCH_MEASURES] for m in ("random", "morbo")},
    }
)
comparison = readable(comparison)
save_table(comparison, "kpi_comparison")
display(comparison)

results = readable(compare.method_table(runs).drop(columns=["run"]))
save_table(results, "method_results")
results


**Observations.** Against today, MORBO's recommendation is better on both searched objectives, coverage (0.6150
against 0.5743) and separation (0.6224 against 0.6118), and on the recorded throughput objective (3.3018 against
3.2862). On the seven network KPIs it improves four and worsens two:
- the hole rate falls from 0.1149 to 0.1137 and the weak rate from 0.3450 to 0.2956;
- median throughput rises from 44.4 to 46.0 Mbit/s and mean throughput from 73.9 to 76.0 Mbit/s;
- the co-band overlap rate rises from 0.3072 to 0.3254 and overlap neighbours per covered tile from 0.959 to 0.962;
- the 5th-percentile throughput stays at 0.

Random search's recommendation has slightly lower hole and weak rates and a higher median throughput, but a lower
mean throughput and far more overlap (0.3880, 1.111 neighbours), and its separation falls below today's.

It took 4.92 minutes of wall clock, of which 2.64 were ray tracing (2.2 s per evaluation); GP fitting and acquisition
make up the rest. Random search took 3.74 minutes, 3.10 of them ray tracing on the same GPU (2.5 s per evaluation),
so the per-evaluation ray-tracing time varies with GPU state, not with the method.

## 9. Coverage Map Comparison

Best-server RSRP before and after MORBO, and the tiles that crossed the coverage-hole threshold; then the RSRP change
of each method's recommended configuration on one scale. Crimson triangles are the masts.


In [ ]:
best_rsrp = {"incumbent": max_rsrp(baseline["rsrp_dbm"].astype(float))}
best_rsrp.update({m: max_rsrp(best[m].radio_map["rsrp_dbm"].astype(float)) for m in METHODS})

figure = plots.coverage_maps(
    best_rsrp["incumbent"], best_rsrp["morbo"], baseline, cfg, sectors=sectors, name="morbo"
)
save_fig(figure, "coverage_before_after_morbo")
plt.show()

# No path in either map makes the change undefined; it stays blank.
with np.errstate(invalid="ignore"):
    change = {label(m): best_rsrp[m] - best_rsrp["incumbent"] for m in METHODS}
figure = plots.map_row(
    change, baseline, colorbar_label="Change in best-server RSRP [dB]", symmetric=True, sectors=sectors
)
save_fig(figure, "rsrp_change_maps")
plt.show()


**Observations.** Over all bands, MORBO raises best-server RSRP on 81.6 % of the tiles reached in both maps, by a
median of 2.3 dB, and closes 440 hole tiles while opening 317. Random search raises 79.1 % by a median of 3.4 dB and
closes 670 while opening 401. (These all-band figures were computed for the commentary from the archived maps; the
maps above are per band.)

## 10. KPI-Level Analysis

Coverage classes by area and by demand, overlap, and which band serves the UEs, for the current configuration and
each method's recommended one. Demand is the incumbent's for every configuration, so the weights do not move.


In [ ]:
configurations = {"incumbent": compare.configuration(baseline, ue, cfg)}
configurations.update({m: compare.configuration(best[m].radio_map, ue, cfg) for m in METHODS})
coverage = readable(compare.coverage_by_area_and_demand(configurations, cfg))
save_table(coverage, "coverage_by_area_and_demand")
display(coverage)

overlap = readable(compare.overlap_neighbour_summary(configurations, cfg))
save_table(overlap, "overlap")
display(overlap)

band_labels = [str(band) for band in baseline["band_label"]]
service = {name: compare.service_summary(c.served, band_labels) for name, c in configurations.items()}
service_table = readable(pd.DataFrame(service).T.rename_axis("configuration").reset_index())
save_table(service_table, "ue_service_summary")
display(service_table)
figure = plots.band_share_bars(service, band_labels)
save_fig(figure, "serving_band_mix")
plt.show()

# Every reported KPI, over all bands and per band, through the same functions:
# a band row is the definition given one band's layers, nothing new.
per_band = compare.band_kpis(configurations, band_labels, cfg)
save_table(readable(per_band), "band_kpis")
display(readable(per_band))
figure = plots.band_kpi_panels(
    per_band,
    ("hole_rate", "weak_rate", "overlap_rate", "rsrp_p05_dbm", "sinr_p05_db"),
)
save_fig(figure, "band_kpis")
plt.show()


### Coverage holes

MORBO cuts the hole rate by area from 0.1149 to 0.1137; random search to 0.1123. By UE reports, the share on hole
tiles, served by nothing at 0 Mbit/s, stays at 21.2 % under MORBO (21.3 %) and random search (21.1 %). Most of it is
one hotspot no tilt reaches.

### Coverage overlap

**Neither recommendation lowers the band-collapsed overlap rate**: 0.3072 rises to 0.3254 under MORBO and 0.3880
under random search. Per band, MORBO lowers it on every layer — 2600 MHz 0.1930 to 0.1884, 1800 MHz 0.2069 to
0.2028, 700 MHz 0.2435 to 0.2236 — yet the collapsed rate rises, because the crowded tiles of the three layers
coincide less. It leaves fewer covered tiles in a three-or-more pile-up (16.1 % against 17.8 %) but more with one or
two neighbours, so the mean barely moves, 0.959 to 0.962. Of MORBO's other front points, evaluation 69 has the
lowest overlap rate, 0.3198, and evaluation 27 the fewest neighbours per covered tile, 0.956.

### Multi-band coordination

**MORBO moves traffic off 2600 MHz onto the lower bands.** 2600 MHz falls from 56.6 % of UE reports to 50.6 %,
1800 MHz rises from 11.3 % to 13.9 % and 700 MHz from 10.9 % to 14.2 %. Median served SINR rises on 2600 MHz (10.9 to
12.9 dB) and 700 MHz (20.0 to 20.7 dB) and falls on 1800 MHz (14.3 to 12.9 dB). Random search moves more traffic onto
1800 MHz and lowers served SINR on 700 MHz.

### Weak coverage

Weak coverage falls by area from 34.5 % to 29.6 % under MORBO and to 29.4 % under random search. By UE reports, the
share on good tiles rises from 51.8 % to 54.2 % under MORBO and 55.4 % under random search.

## 11. Key Observations

- **MORBO finds what random search does not.** 10 of its configurations beat today on both searched objectives; none
  of random search's does. Its final hypervolume is 0.4075 against 0.3754, and it passes random search's final value
  at evaluation 14 of 73.
- **The recommendation beats today on both objectives.** Evaluation 21 has the best separation on the front and a
  contribution five times the next point's.
- **The recommendation reshapes the low band** (700 MHz uptilted 2.7° on average) and moves load off 2600 MHz onto
  the two lower bands, with higher served SINR on 2600 and 700 MHz.
- **Overlap is the weak spot.** Per-band overlap falls on every layer, but no candidate of either search beats
  today's band-collapsed overlap rate.
- **These are one seed on one scenario;** notebook 04 reports intervals over seeds when more than one is run.

## 12. Limitations

- **The recommendation depends on the reference point.** Against the origin, contributions are tiny and an extreme point
  can win by a sliver; a reference at the incumbent, or a pick restricted to points that beat today, would choose
  differently here.
- **Budget.** 73 evaluations in 36 dimensions; MORBO's hypervolume was still rising at evaluation 70, and the trust
  region never halved, so neither the shrinking nor the restart was exercised on this run.
- **One trust region.** `trust_region.count` is 1; the paper's default of 5 was not tried.
- **Not bit-reproducible.** The GP fits can amplify GPU round-off into different proposals; the objectives are rounded to
  6 significant digits before the search reads them, which makes a mismatch between reruns unlikely, not impossible.
- **GP prior.** The local GPs carry BoTorch's dimension-scaled lengthscale prior, which the paper does not use; without
  it the first ray-traced run failed to fit.
- **The objectives disagree with the overlap KPI.** Separation rises on most of the front while no candidate beats
  today's band-collapsed overlap rate: one is a soft per-tile product over bands, the other counts any crowded band.
- **Winner's curse.** Every candidate is scored under one solver seed, so the recommendation's scores are biased upward
  (notebook 04, threats to validity).
- **No movement constraint.** Tilt change is reported, not penalised, so the proposal moves 32 of 36 antennas, one by
  9.9°.

## 13. Conclusion

MORBO is a working model-guided multi-objective search for multi-band tilt on this scenario. It:

- reaches a larger hypervolume than random search at the same budget and seed, from the same Sobol design;
- finds 10 configurations better than today on both coverage and separation, where random search finds none;
- recommends, under the chosen rule, a configuration that gains coverage, separation, holes, weak coverage and
  throughput, and loses a little co-band overlap; front points with more coverage sit beside it in the shortlist.

Whether MORBO's margin holds is decided over seeds in notebook 04.